# Portfolio Risk Analysis

This notebook demonstrates comprehensive portfolio risk analysis techniques:
- **Value at Risk (VaR)**: Historical, parametric, and Cornish-Fisher methods
- **Expected Shortfall**: Conditional VaR for tail risk
- **Sharpe Ratio**: Risk-adjusted performance
- **Maximum Drawdown**: Peak-to-trough decline
- **Correlation Analysis**: Portfolio diversification
- **Stress Testing**: Scenario-based risk assessment

These metrics are essential for risk management and portfolio construction.

In [ ]:
# Import required libraries
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Import custom modules
from utils import generate_price_series
from risk import (
    calculate_var,
    calculate_expected_shortfall,
    calculate_portfolio_var,
    calculate_sharpe_ratio,
    calculate_max_drawdown,
    calculate_volatility,
    calculate_portfolio_risk_metrics,
    stress_test_portfolio,
    calculate_correlation_matrix
)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print('✓ Libraries loaded successfully')

## 1. Generate Portfolio Data

Create synthetic returns for a multi-asset portfolio.

In [ ]:
# Generate price series for multiple assets
np.random.seed(42)
n_periods = 1000
n_assets = 5

# Generate correlated asset returns
asset_names = ['Asset_A', 'Asset_B', 'Asset_C', 'Asset_D', 'Asset_E']
returns_dict = {}

# Create correlation structure
for i, name in enumerate(asset_names):
    prices = generate_price_series(
        n_periods=n_periods,
        initial_price=100.0,
        mu=0.0005 * (1 + i * 0.1),  # Different expected returns
        sigma=0.015 * (1 + i * 0.15),  # Different volatilities
        seed=42 + i
    )
    returns_dict[name] = prices.pct_change().dropna()

# Combine into DataFrame
returns = pd.DataFrame(returns_dict)

# Portfolio weights
weights = np.array([0.3, 0.25, 0.2, 0.15, 0.1])

# Calculate portfolio returns
portfolio_returns = (returns * weights).sum(axis=1)

print(f'Generated {len(returns)} periods of returns for {n_assets} assets')
print(f'\nPortfolio weights:')
for name, weight in zip(asset_names, weights):
    print(f'  {name}: {weight:.1%}')
print(f'\nReturns summary:')
print(returns.describe())

## 2. Visualize Portfolio Returns

Examine the distribution and evolution of portfolio returns.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Cumulative returns
cumulative_returns = (1 + returns).cumprod()
portfolio_cumulative = (1 + portfolio_returns).cumprod()

axes[0, 0].plot(cumulative_returns.index, cumulative_returns, alpha=0.7, linewidth=1)
axes[0, 0].plot(portfolio_cumulative.index, portfolio_cumulative, 
                color='black', linewidth=2, label='Portfolio')
axes[0, 0].set_ylabel('Cumulative Return')
axes[0, 0].set_title('Cumulative Returns')
axes[0, 0].legend(asset_names + ['Portfolio'], loc='best')
axes[0, 0].grid(True, alpha=0.3)

# Portfolio returns distribution
axes[0, 1].hist(portfolio_returns, bins=50, alpha=0.7, edgecolor='black', density=True)
axes[0, 1].axvline(portfolio_returns.mean(), color='r', linestyle='--', 
                   linewidth=2, label=f'Mean: {portfolio_returns.mean():.4f}')
axes[0, 1].axvline(portfolio_returns.median(), color='g', linestyle='--', 
                   linewidth=2, label=f'Median: {portfolio_returns.median():.4f}')
# Add normal distribution overlay
mu, sigma = portfolio_returns.mean(), portfolio_returns.std()
x = np.linspace(portfolio_returns.min(), portfolio_returns.max(), 100)
axes[0, 1].plot(x, stats.norm.pdf(x, mu, sigma), 'k--', linewidth=2, label='Normal dist')
axes[0, 1].set_xlabel('Returns')
axes[0, 1].set_ylabel('Density')
axes[0, 1].set_title('Portfolio Returns Distribution')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Rolling volatility
rolling_vol = calculate_volatility(portfolio_returns, window=50, annualize=True, trading_days=252)
axes[1, 0].plot(rolling_vol.index, rolling_vol, linewidth=1.5, color='blue')
axes[1, 0].axhline(rolling_vol.mean(), color='r', linestyle='--', 
                   label=f'Mean: {rolling_vol.mean():.2%}')
axes[1, 0].set_ylabel('Annualized Volatility')
axes[1, 0].set_xlabel('Time')
axes[1, 0].set_title('Rolling Volatility (50-period)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Q-Q plot
stats.probplot(portfolio_returns, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot (Normal Distribution)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('✓ Portfolio returns visualization complete')

## 3. Value at Risk (VaR)

Calculate VaR using different methods to estimate potential losses.

In [ ]:
# Calculate VaR at different confidence levels
confidence_levels = [0.90, 0.95, 0.99]
methods = ['historical', 'parametric', 'cornish_fisher']

var_results = []
for method in methods:
    for confidence in confidence_levels:
        var = calculate_var(portfolio_returns, confidence, method)
        var_results.append({
            'Method': method.capitalize(),
            'Confidence': f'{confidence:.0%}',
            'VaR': var,
            'VaR (%)': var * 100
        })

var_df = pd.DataFrame(var_results)

print('Value at Risk (VaR) Analysis:')
print('=' * 70)
print(var_df.to_string(index=False))
print('=' * 70)
print('\nInterpretation: VaR represents the maximum expected loss at given confidence level')
print('Example: 95% VaR of 2.5% means 95% confidence losses won\'t exceed 2.5%')

In [ ]:
# Visualize VaR
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# VaR by method
pivot_df = var_df.pivot(index='Confidence', columns='Method', values='VaR (%)')
pivot_df.plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_ylabel('VaR (%)')
axes[0].set_title('Value at Risk by Method and Confidence Level')
axes[0].legend(title='Method')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# VaR on distribution
axes[1].hist(portfolio_returns * 100, bins=50, alpha=0.7, edgecolor='black', density=True)
var_95 = calculate_var(portfolio_returns, 0.95, 'historical')
var_99 = calculate_var(portfolio_returns, 0.99, 'historical')
axes[1].axvline(-var_95 * 100, color='orange', linestyle='--', linewidth=2, label='95% VaR')
axes[1].axvline(-var_99 * 100, color='red', linestyle='--', linewidth=2, label='99% VaR')
axes[1].set_xlabel('Returns (%)')
axes[1].set_ylabel('Density')
axes[1].set_title('VaR Thresholds on Returns Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Expected Shortfall (CVaR)

Calculate Expected Shortfall, which measures tail risk beyond VaR.

In [ ]:
# Calculate Expected Shortfall
es_results = []
for confidence in confidence_levels:
    var = calculate_var(portfolio_returns, confidence, 'historical')
    es = calculate_expected_shortfall(portfolio_returns, confidence)
    es_results.append({
        'Confidence': f'{confidence:.0%}',
        'VaR (%)': var * 100,
        'Expected Shortfall (%)': es * 100,
        'ES/VaR Ratio': es / var if var != 0 else 0
    })

es_df = pd.DataFrame(es_results)

print('Expected Shortfall (CVaR) Analysis:')
print('=' * 70)
print(es_df.to_string(index=False))
print('=' * 70)
print('\nInterpretation: Expected Shortfall is the average loss given that VaR is exceeded')
print('ES provides a more comprehensive tail risk measure than VaR')
print('ES/VaR > 1 indicates heavy tails (extreme losses beyond VaR)')

In [ ]:
# Visualize VaR vs Expected Shortfall
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(es_df))
width = 0.35

bars1 = ax.bar(x - width/2, es_df['VaR (%)'], width, label='VaR', alpha=0.8)
bars2 = ax.bar(x + width/2, es_df['Expected Shortfall (%)'], width, label='Expected Shortfall', alpha=0.8)

ax.set_xlabel('Confidence Level')
ax.set_ylabel('Risk Measure (%)')
ax.set_title('Value at Risk vs Expected Shortfall')
ax.set_xticks(x)
ax.set_xticklabels(es_df['Confidence'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Sharpe Ratio and Risk-Adjusted Performance

Calculate Sharpe ratio and other risk-adjusted performance metrics.

In [ ]:
# Calculate Sharpe ratio for each asset and portfolio
risk_free_rate = 0.02  # 2% annual risk-free rate

sharpe_ratios = {}
for col in returns.columns:
    sharpe_ratios[col] = calculate_sharpe_ratio(returns[col], risk_free_rate)

sharpe_ratios['Portfolio'] = calculate_sharpe_ratio(portfolio_returns, risk_free_rate)

# Calculate annualized metrics
annualized_return = portfolio_returns.mean() * 252
annualized_vol = calculate_volatility(portfolio_returns, window=None, annualize=True, trading_days=252)

print('Risk-Adjusted Performance Metrics:')
print('=' * 70)
print('\nSharpe Ratios:')
for name, sharpe in sharpe_ratios.items():
    print(f'  {name:15s}: {sharpe:.4f}')

print(f'\nPortfolio Statistics (Annualized):')
print(f'  Expected Return: {annualized_return:.2%}')
print(f'  Volatility: {annualized_vol:.2%}')
print(f'  Sharpe Ratio: {sharpe_ratios["Portfolio"]:.4f}')
print(f'  Risk-free rate: {risk_free_rate:.2%}')
print('\nInterpretation: Sharpe ratio > 1 is generally considered good')

In [ ]:
# Visualize risk-return profile
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Risk-return scatter
for col in returns.columns:
    ret = returns[col].mean() * 252 * 100
    vol = returns[col].std() * np.sqrt(252) * 100
    axes[0].scatter(vol, ret, s=100, alpha=0.6, label=col)

# Add portfolio
port_ret = annualized_return * 100
port_vol = annualized_vol * 100
axes[0].scatter(port_vol, port_ret, s=200, color='red', marker='*', 
                edgecolors='black', linewidths=2, label='Portfolio', zorder=10)

axes[0].set_xlabel('Annualized Volatility (%)')
axes[0].set_ylabel('Annualized Return (%)')
axes[0].set_title('Risk-Return Profile')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sharpe ratios comparison
names = list(sharpe_ratios.keys())
values = list(sharpe_ratios.values())
colors = ['blue'] * (len(names) - 1) + ['red']
axes[1].bar(names, values, color=colors, alpha=0.7, edgecolor='black')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axhline(1, color='green', linestyle='--', alpha=0.5, label='Sharpe = 1')
axes[1].set_ylabel('Sharpe Ratio')
axes[1].set_title('Sharpe Ratio Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 6. Maximum Drawdown Analysis

Calculate maximum drawdown and recovery periods.

In [ ]:
# Calculate portfolio value
initial_value = 1000000  # $1M
portfolio_value = initial_value * portfolio_cumulative

# Calculate drawdown metrics
dd_metrics = calculate_max_drawdown(portfolio_value)

print('Maximum Drawdown Analysis:')
print('=' * 70)
print(f'Maximum Drawdown: {dd_metrics["max_drawdown"]:.2%}')
print(f'Peak Date: {dd_metrics["peak_date"]}')
print(f'Trough Date: {dd_metrics["max_drawdown_date"]}')
if dd_metrics['recovery_date']:
    print(f'Recovery Date: {dd_metrics["recovery_date"]}')
    print(f'Recovery Period: {dd_metrics["recovery_days"]} days')
else:
    print('Recovery: Not yet recovered')
print('=' * 70)

In [ ]:
# Visualize drawdown
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Portfolio value with drawdown period
axes[0].plot(portfolio_value.index, portfolio_value, linewidth=1.5, label='Portfolio Value')
running_max = portfolio_value.expanding().max()
axes[0].plot(running_max.index, running_max, linewidth=1, linestyle='--', 
             alpha=0.7, color='green', label='Running Maximum')
axes[0].axvline(dd_metrics['peak_date'], color='green', linestyle='--', alpha=0.5, label='Peak')
axes[0].axvline(dd_metrics['max_drawdown_date'], color='red', linestyle='--', alpha=0.5, label='Trough')
if dd_metrics['recovery_date']:
    axes[0].axvline(dd_metrics['recovery_date'], color='blue', linestyle='--', alpha=0.5, label='Recovery')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title(f'Portfolio Value and Maximum Drawdown ({dd_metrics["max_drawdown"]:.2%})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y:,.0f}'))

# Drawdown series
drawdown = (portfolio_value - running_max) / running_max
axes[1].fill_between(drawdown.index, 0, drawdown * 100, alpha=0.5, color='red')
axes[1].plot(drawdown.index, drawdown * 100, linewidth=1, color='darkred')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Time')
axes[1].set_title('Underwater Plot (Drawdown from Peak)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Correlation Analysis

Analyze portfolio diversification through correlation.

In [ ]:
# Calculate correlation matrix
correlation_matrix = calculate_correlation_matrix(returns)

print('Correlation Matrix:')
print('=' * 70)
print(correlation_matrix.round(3))
print('=' * 70)

# Calculate average correlation
n = len(correlation_matrix)
avg_corr = (correlation_matrix.sum().sum() - n) / (n * (n - 1))
print(f'\nAverage pairwise correlation: {avg_corr:.3f}')
print('\nInterpretation:')
print('  Correlation close to 0: Assets move independently (good diversification)')
print('  Correlation close to 1: Assets move together (poor diversification)')
print('  Correlation close to -1: Assets move oppositely (excellent diversification)')

In [ ]:
# Visualize correlation matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8},
            ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title('Asset Correlation Matrix', fontsize=14)

# Rolling correlation (Asset_A vs Asset_B)
rolling_corr = returns['Asset_A'].rolling(window=100).corr(returns['Asset_B'])
axes[1].plot(rolling_corr.index, rolling_corr, linewidth=1.5)
axes[1].axhline(rolling_corr.mean(), color='r', linestyle='--', 
                label=f'Mean: {rolling_corr.mean():.3f}')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('Correlation')
axes[1].set_xlabel('Time')
axes[1].set_title('Rolling Correlation: Asset_A vs Asset_B (100-period)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Stress Testing

Evaluate portfolio performance under extreme scenarios.

In [ ]:
# Define stress test scenarios
scenarios = {
    'Market Crash': {'Asset_A': -0.20, 'Asset_B': -0.18, 'Asset_C': -0.22, 'Asset_D': -0.15, 'Asset_E': -0.25},
    'Tech Selloff': {'Asset_A': -0.30, 'Asset_B': -0.05, 'Asset_C': -0.10, 'Asset_D': -0.02, 'Asset_E': -0.28},
    'Financial Crisis': {'Asset_A': -0.15, 'Asset_B': -0.35, 'Asset_C': -0.30, 'Asset_D': -0.20, 'Asset_E': -0.12},
    'Inflation Shock': {'Asset_A': -0.10, 'Asset_B': -0.12, 'Asset_C': -0.08, 'Asset_D': -0.15, 'Asset_E': -0.05},
    'Black Swan': {'Asset_A': -0.40, 'Asset_B': -0.45, 'Asset_C': -0.38, 'Asset_D': -0.42, 'Asset_E': -0.50}
}

# Run stress tests
stress_results = stress_test_portfolio(returns, weights, scenarios)

# Calculate dollar impact
current_portfolio_value = portfolio_value.iloc[-1]
stress_results['Dollar Loss'] = stress_results['portfolio_return'] * current_portfolio_value
stress_results['Percent Loss'] = stress_results['portfolio_return'] * 100

print('Stress Test Results:')
print('=' * 70)
print(f'Current Portfolio Value: ${current_portfolio_value:,.2f}')
print()
print(stress_results[['scenario', 'Percent Loss', 'Dollar Loss']].to_string(index=False))
print('=' * 70)
print('\nWorst scenario:', stress_results.loc[stress_results['portfolio_return'].idxmin(), 'scenario'])
print('Best scenario:', stress_results.loc[stress_results['portfolio_return'].idxmax(), 'scenario'])

In [ ]:
# Visualize stress test results
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['red' if x < -0.30 else 'orange' if x < -0.15 else 'yellow' 
          for x in stress_results['portfolio_return']]
bars = ax.barh(stress_results['scenario'], stress_results['Percent Loss'], 
               color=colors, alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Portfolio Loss (%)')
ax.set_title('Stress Test Results - Portfolio Losses by Scenario')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(width - 1, bar.get_y() + bar.get_height()/2, 
            f'{width:.1f}%', ha='right', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Comprehensive Risk Summary

Consolidate all risk metrics into a comprehensive report.

In [ ]:
# Calculate comprehensive risk metrics
comprehensive_metrics = calculate_portfolio_risk_metrics(returns, weights)

# Create summary report
risk_summary = pd.DataFrame({
    'Metric': [
        'Annualized Return',
        'Annualized Volatility',
        'Sharpe Ratio',
        'Maximum Drawdown',
        'Value at Risk (95%)',
        'Value at Risk (99%)',
        'Expected Shortfall (95%)',
        'Current Portfolio Value'
    ],
    'Value': [
        f'{annualized_return:.2%}',
        f'{comprehensive_metrics["volatility"]:.2%}',
        f'{comprehensive_metrics["sharpe_ratio"]:.4f}',
        f'{comprehensive_metrics["max_drawdown"]:.2%}',
        f'{comprehensive_metrics["var_95"]*100:.2f}%',
        f'{comprehensive_metrics["var_99"]*100:.2f}%',
        f'{comprehensive_metrics["expected_shortfall_95"]*100:.2f}%',
        f'${portfolio_value.iloc[-1]:,.2f}'
    ],
    'Interpretation': [
        'Expected annual return',
        'Annual return variability',
        'Risk-adjusted performance',
        'Largest peak-to-trough decline',
        'Max 1-day loss (95% confidence)',
        'Max 1-day loss (99% confidence)',
        'Average loss beyond 95% VaR',
        'Current portfolio worth'
    ]
})

print('\n' + '=' * 80)
print('COMPREHENSIVE PORTFOLIO RISK REPORT')
print('=' * 80)
print(risk_summary.to_string(index=False))
print('=' * 80)

# Risk grade
risk_score = 0
if comprehensive_metrics['sharpe_ratio'] > 1: risk_score += 2
elif comprehensive_metrics['sharpe_ratio'] > 0.5: risk_score += 1
if comprehensive_metrics['volatility'] < 0.15: risk_score += 2
elif comprehensive_metrics['volatility'] < 0.25: risk_score += 1
if abs(comprehensive_metrics['max_drawdown']) < 0.15: risk_score += 2
elif abs(comprehensive_metrics['max_drawdown']) < 0.25: risk_score += 1

grade_map = {6: 'A (Excellent)', 5: 'B (Good)', 4: 'C (Fair)', 3: 'D (Moderate)', 2: 'E (High)', 1: 'F (Very High)', 0: 'F (Extreme)'}
risk_grade = grade_map.get(risk_score, 'F (Extreme)')

print(f'\nRisk Grade: {risk_grade}')
print('\nKey Observations:')
print(f'  • Portfolio generates {annualized_return:.2%} annual return with {comprehensive_metrics["volatility"]:.2%} volatility')
print(f'  • Sharpe ratio of {comprehensive_metrics["sharpe_ratio"]:.2f} indicates {"good" if comprehensive_metrics["sharpe_ratio"] > 1 else "moderate"} risk-adjusted returns')
print(f'  • Maximum drawdown of {comprehensive_metrics["max_drawdown"]:.2%} shows {"acceptable" if abs(comprehensive_metrics["max_drawdown"]) < 0.20 else "elevated"} downside risk')
print(f'  • 95% VaR of {comprehensive_metrics["var_95"]*100:.2f}% means potential daily loss exceeds this in 5% of cases')

## Conclusion

In this notebook, we've performed comprehensive portfolio risk analysis:

### Risk Metrics Covered:

**1. Value at Risk (VaR)**
- Historical: Based on actual distribution
- Parametric: Assumes normal distribution
- Cornish-Fisher: Accounts for skewness and kurtosis

**2. Expected Shortfall (CVaR)**
- Measures average loss beyond VaR threshold
- More comprehensive tail risk measure
- Useful for regulatory capital requirements

**3. Sharpe Ratio**
- Risk-adjusted performance metric
- Considers excess return per unit of risk
- Enables comparison across different strategies

**4. Maximum Drawdown**
- Peak-to-trough decline
- Important for capital preservation
- Indicates worst historical loss

**5. Correlation Analysis**
- Measures diversification benefits
- Identifies concentration risks
- Time-varying correlations reveal regime changes

**6. Stress Testing**
- Scenario-based risk assessment
- Evaluates extreme market conditions
- Critical for risk management planning

### Key Takeaways:
- Multiple risk metrics provide comprehensive view
- Diversification reduces but doesn't eliminate risk
- Tail risk (VaR, ES) often more important than average volatility
- Regular monitoring and stress testing essential

### Next Steps:
- Implement dynamic risk limits and alerts
- Develop portfolio optimization strategies
- Explore optimal execution strategies (notebook 06)